## Importing and Initial Variables

In [1]:
# from todoist_api_python.api import TodoistAPI
import pandas as pd
from utils import credentials as cd
import requests
from datetime import date, datetime
from pathlib import Path

FILENAME = "my-energysystem-tasks.csv"
mostrecentdate = None
data_path = Path(Path.cwd(),r"data/",FILENAME)

In [6]:
Path.exists(data_path)

False

## Check for existing archive of tasks

In [9]:
# Path.read_bytes(data_path)
if Path.exists(data_path) is False:
    print("file not found.\npull from earliest month start.")
else:
    print("file found. Opening to collect last task date")
    df = pd.read_csv(data_path)
    mostrecentdate = df['completed_at'].max()
    del df

file not found.
pull from earliest month start.


### Grab the last completed date as a starting point to pull more tasks

In [10]:
if mostrecentdate is None:
    print("No most recent date found")
    start_date = datetime.now()
    start_date = datetime.replace(start_date,day=1)
    end_date = datetime.now()
    
    iso_start_date = start_date.isoformat()
    iso_end_date = end_date.isoformat()
    # datetime.strptime("5/1/2026","%m/%d/%Y")

# end_date = datetime.strptime("5/18/2026","%m/%d/%Y")



No most recent date found


## Prep API call to collect tasks

In [11]:
url = "https://api.todoist.com/api/v1/tasks/completed/by_completion_date/"

headers = {"Authorization": "Bearer " + cd.api_key}

request_body = {
    "since" : iso_start_date,
    "until" : iso_end_date,
    "limit": 200
}

In [12]:
response = requests.get(url, headers=headers,params = request_body)

In [13]:
data = response.json()

items = data.get("items",[])

len(items)

64

## Build Task Table

In [71]:
task_table = pd.DataFrame(items)

In [ ]:
task_table['completed_at'].max()

In [72]:
col = task_table.columns

In [73]:
idx = 0
column_dict = dict()

for v in col.to_list():
    # print(f'adding to dict: Key= {idx} | Value = {v}')
    column_dict[idx] = v
    idx += 1

In [74]:
remove_columns = [1,2,3,5,6,8,15,16,18,22,25]

kept_columns = {k : v for k,v in column_dict.items() if k not in remove_columns}

kept_col_list = list(kept_columns.keys())
kept_col_list

[0, 4, 7, 9, 10, 11, 12, 13, 14, 17, 19, 20, 21, 23, 24]

In [75]:
cleaned_table = task_table.iloc[:,kept_col_list]

# cleaned_table

In [96]:
id_column = (cleaned_table.columns.get_loc("id"), "id")

## Split due date iterable into unique columns

In [90]:
slicer = cleaned_table[[id_column[1],'due']]
slicer = slicer[slicer['due'].notna()]

duedates = slicer['due'].apply(pd.Series)

source_names = duedates.columns.tolist()
updated_names = ["Due" + part.capitalize() for part in source_names]

rename_zip = zip(source_names, updated_names)
rename_dict = dict(rename_zip)


In [77]:
duedates = duedates.rename(rename_dict, axis=1)

combined_df = slicer.join(duedates)
duedates = combined_df.iloc[:,[0,2]]

In [ ]:
cleaned_table = pd.merge(left=task_table,right=duedates, on="id")

In [ ]:
id_column[0]

14

In [109]:
def split_column(data,split):
    test = data[[id_column[1],split]]
    test = test[test.iloc[:, 1].notna()]
    unnest = test.apply(pd.Series)
    source_names = unnest.columns.tolist()
    updated_names = ["Due" + part.capitalize() for part in source_names]

    rename_zip = zip(source_names, updated_names)
    rename_dict = dict(rename_zip)

    renamed = unnest.rename(rename_dict, axis=1)
    output = test.join(renamed)
    return output

x = split_column(cleaned_table,"due")

x

,id,due,DueId,DueDue
0,6ggHvg6XV69Wjwf2,"{'date': '2026-05-18', 'is_recurring': False, ...",6ggHvg6XV69Wjwf2,"{'date': '2026-05-18', 'is_recurring': False, ..."
1,6ggHvhwj6vQc8vw2,"{'date': '2026-05-18', 'is_recurring': False, ...",6ggHvhwj6vQc8vw2,"{'date': '2026-05-18', 'is_recurring': False, ..."
2,6gg5WQrJCW6v3hRR,"{'date': '2026-05-17', 'is_recurring': False, ...",6gg5WQrJCW6v3hRR,"{'date': '2026-05-17', 'is_recurring': False, ..."
3,6gffXwV59HmX3WvR,"{'date': '2026-05-15', 'is_recurring': False, ...",6gffXwV59HmX3WvR,"{'date': '2026-05-15', 'is_recurring': False, ..."
4,6gffXq4PW45F2PC2,"{'date': '2026-05-15', 'is_recurring': False, ...",6gffXq4PW45F2PC2,"{'date': '2026-05-15', 'is_recurring': False, ..."
5,6gHqXFm8RXmQV2xQ,"{'date': '2026-05-14', 'is_recurring': False, ...",6gHqXFm8RXmQV2xQ,"{'date': '2026-05-14', 'is_recurring': False, ..."
6,6gfMqf2Q3wX2W3r2,"{'date': '2026-05-14', 'is_recurring': False, ...",6gfMqf2Q3wX2W3r2,"{'date': '2026-05-14', 'is_recurring': False, ..."
7,6gfJW82mq47R6592,"{'date': '2026-05-14', 'is_recurring': False, ...",6gfJW82mq47R6592,"{'date': '2026-05-14', 'is_recurring': False, ..."
8,6gfJWC5w6wjgh7fR,"{'date': '2026-05-14', 'is_recurring': False, ...",6gfJWC5w6wjgh7fR,"{'date': '2026-05-14', 'is_recurring': False, ..."
9,6XJV7f3jfWW8HCWq,"{'date': '2026-05-13', 'is_recurring': False, ...",6XJV7f3jfWW8HCWq,"{'date': '2026-05-13', 'is_recurring': False, ..."


In [108]:
x.iloc[:,1]

0     {'date': '2026-05-18', 'is_recurring': False, ...
1     {'date': '2026-05-18', 'is_recurring': False, ...
2     {'date': '2026-05-17', 'is_recurring': False, ...
3     {'date': '2026-05-15', 'is_recurring': False, ...
4     {'date': '2026-05-15', 'is_recurring': False, ...
5     {'date': '2026-05-14', 'is_recurring': False, ...
6     {'date': '2026-05-14', 'is_recurring': False, ...
7     {'date': '2026-05-14', 'is_recurring': False, ...
8     {'date': '2026-05-14', 'is_recurring': False, ...
9     {'date': '2026-05-13', 'is_recurring': False, ...
10    {'date': '2026-05-13', 'is_recurring': False, ...
11    {'date': '2026-05-13', 'is_recurring': False, ...
12    {'date': '2026-05-12', 'is_recurring': False, ...
13    {'date': '2026-05-12', 'is_recurring': False, ...
14    {'date': '2026-05-12', 'is_recurring': False, ...
15    {'date': '2026-05-11', 'is_recurring': False, ...
16    {'date': '2026-05-11', 'is_recurring': False, ...
17    {'date': '2026-05-10', 'is_recurring': Fal

In [ ]:
# date_dict = cleaned_table.due.describe()['top']


# for k,v in date_dict.items():
#     # print("Col:{} Val:{}".format(k,v))
#     col_name = "Due"+ k.capitalize()
#     cleaned_table[col_name] = v

# # cleaned_table
# # for key, value in date_dict.items():
# #     print(key)
# #     print(value)

In [ ]:
test_project_id = cleaned_table.iloc[5].project_id